In [1]:
import os
import sys
import getpass
import importlib
import tqdm
usrname = getpass.getuser()
sys.path.append('/Users/aghavamp/Desktop/Projects')
sys.path.append('/Users/aghavamp/Desktop/Projects/Functional_Fusion')
sys.path.append('/Users/aghavamp/Desktop/Projects/SUITPy')
sys.path.append('/Users/aghavamp/Desktop/Projects/AnatSearchlight')
sys.path.append(f'/Users/{usrname}/Desktop/Projects/PcmPy')

import AnatSearchlight.searchlight as sl
import scipy.io as sio
import rsatoolbox as rsa
from rsatoolbox.io import spm as spm_io
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import pandas as pd
import surfAnalysisPy as surf
import SUITPy as suit
import nibabel as nb
from nibabel import cifti2
import nitools as nt
from matplotlib.cm import ScalarMappable
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from pathlib import Path
import seaborn as sns
import PcmPy as pcm
import Functional_Fusion.atlas_map as am
import Functional_Fusion.reliability as rel
import glob
import matplotlib.patches as patches
from nilearn.plotting.cm import _cmap_d as nilearn_cmaps

# SET PATHS:
baseDir = os.path.join('/Users', getpass.getuser(), 'Desktop', 'Projects', 'bimanual_wrist', 'data', 'fMRI')
bidsDir = 'BIDS'
anatomicalDir = 'anatomicals'
freesurferDir = 'surfaceFreesurfer'
surfacewbDir = 'surfaceWB' 
behavDir = 'behavioural'
regDir = 'ROI'
atlasDir = '/Volumes/diedrichsen_data$/data/Atlas_templates/fs_LR_32'
analysisDir = os.path.join(os.path.dirname(os.path.dirname(baseDir)), 'analysis')



There is the congruency effect and the bimanual-specific patterns that could derive the bimanual interaction term in our models. Here within each searchlight, we fit a full model with 'contra' + 'congruency' + 'bimanual-specific' terms. Then we will make the map corresponding to the 'congruency' and 'bimanual-specific' components. This allows us to visually see where exactly we have 'bimanual-specific' encoding vs. just congruency. This complements the full ROI analysis. 

### congruency component

In [5]:
sn_list = [101, 104, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115]
glm = 1
hem = ['L', 'R']
sl_name = ['left_cortex', 'right_cortex']

def model_bimanual_interaction(data, **kwargs):
    Y_all = data[:-1,:] / np.sqrt(data[-1,:]) # univariate prewhitening
    part_vec = kwargs['partition_vec']
    cond_vec = kwargs['condition_vec']
    G_contra = kwargs['G_contra']
    G_incong = kwargs['G_incong']
    G_bimanual = kwargs['G_bimanual']

    # make PCM dataset object:
    rows_bimanual = [j for j, c in enumerate(cond_vec) if c>11]
    Y_bimanual = Y_all[rows_bimanual,:]
    cond_vec_bimanual = cond_vec[rows_bimanual]
    part_vec_bimanual = part_vec[rows_bimanual]

    G_hat,_ = pcm.est_G_crossval(Y_bimanual,
                                cond_vec_bimanual,
                                part_vec_bimanual,
                                X=pcm.matrix.indicator(part_vec_bimanual))

    # OLS:
    X = np.vstack([
        G_contra.flatten(),
        G_incong.flatten(),
        G_bimanual.flatten(),
    ]).T
    pinvX = np.linalg.pinv(X)
    W = pinvX @ G_hat.flatten()

    # OLS variances:
    ols_contra = W[0] * np.trace(G_contra)/36
    ols_congruency = W[1] * np.trace(G_incong)/36
    ols_bimanual = W[2] * np.trace(G_bimanual)/36

    d_contra = 2/(36-1) * ols_contra*36
    d_congruency = 2/(36-1) * ols_congruency*36
    d_bimanual = 2/(36-1) * ols_bimanual*36

    return d_congruency

labels = ['flx_flx',    'flx_flxup',   'flx_extup',   'flx_ext',   'flx_extdn',   'flx_flxdn',
          'flxup_flx',  'flxup_flxup', 'flxup_extup', 'flxup_ext', 'flxup_extdn', 'flxup_flxdn',
          'extup_flx',  'extup_flxup', 'extup_extup', 'extup_ext', 'extup_extdn', 'extup_flxdn',
          'ext_flx',    'ext_flxup',   'ext_extup',   'ext_ext',   'ext_extdn',   'ext_flxdn',
          'extdn_flx',  'extdn_flxup', 'extdn_extup', 'extdn_ext', 'extdn_extdn', 'extdn_flxdn',
          'flxdn_flx',  'flxdn_flxup', 'flxdn_extup', 'flxdn_ext', 'flxdn_extdn', 'flxdn_flxdn']   
F_contra = np.zeros((36,6))
cnd2idx = {'flx':0, 'flxup':1, 'extup':2, 'ext':3, 'extdn':4, 'flxdn':5}
for i in range(36):
    cond_pair = labels[i].split('_')
    cnd_contra = cond_pair[0]
    cnd_ipsi = cond_pair[1]
    idx_contra = cnd2idx[cnd_contra]
    idx_ipsi = cnd2idx[cnd_ipsi]
    F_contra[i, idx_contra] = 1
# center the features:
F_contra -= np.mean(F_contra, axis=0)

# load unimanual G matrix of all regions:
G_uni_df = pd.read_pickle(os.path.join(analysisDir, 'G_uni.pkl'))
# average across all subjects and regions:
G_all = np.concatenate(G_uni_df['G_hat'].tolist(), axis=0)  # (n_regions * N, 6, 6)
Guni = G_all.mean(axis=0)

# build fixed 36x36 model components from group-averaged unimanual G:
G_contra = F_contra @ Guni @ F_contra.T
G_contra = G_contra / np.trace(np.abs(G_contra))

# incongruency model:
ncond = 36
cov_incong = np.zeros((ncond,ncond)) #np.eye(ncond)
idx_incong = [1,2,4,5, 6,9,10,11, 12,15,16,17, 19,20,22,23, 24,25,26,27, 30,31,32,33]
for i in range(len(idx_incong)):
    for j in range(len(idx_incong)):
        if i != j:
            cov_incong[idx_incong[i], idx_incong[j]] = 1
F_incong = np.zeros((ncond,1))
F_incong[idx_incong,0] = 1
cov_incong = F_incong @ F_incong.T
G_incong = pcm.centering(ncond) @ cov_incong @ pcm.centering(ncond) # incongruency model
G_incong = G_incong / np.trace(G_incong) # normalize the overall variance

# bimanual-specific model:
I = np.eye(ncond)
G_bimanual = pcm.centering(ncond) @ I @ I.T @ pcm.centering(ncond) # non-orthogonalized interaction
G_bimanual = G_bimanual / np.trace(G_bimanual) # normalize the overall variance

# orthogonalize the bimanual-specific model:
X = np.c_[G_contra, G_incong]
Io = I - X @ np.linalg.pinv(X) @ I
G_bimanual = pcm.centering(ncond) @ Io @ Io.T @ pcm.centering(ncond) 
G_bimanual = G_bimanual / np.trace(G_bimanual)

for sn in sn_list:
    # try:
    # run marginal MVPA:
    # regressor info:
    reginfo = pd.read_table(os.path.join(baseDir, f'glm{glm}', f's{sn}', 'reginfo.tsv'))
    # get indices of the regressors of interest:
    # regressors_of_interest = reginfo[reginfo.name.str.contains('bi')].index.tolist()
    regressors_of_interest = reginfo.index.tolist()
    
    datafiles = [os.path.join(baseDir, f'glm{glm}', f's{sn}', f'beta_{i+1:04d}.nii') for i in regressors_of_interest]
    datafiles.append(os.path.join(baseDir, f'glm{glm}', f's{sn}', 'ResMS.nii'))
    partition_vec = reginfo.loc[regressors_of_interest, 'run'].values
    condition_vec = reginfo.loc[regressors_of_interest, 'name'].values

    for i, h in enumerate(hem):
        print(f'====================== Processing subject {sn}, hemisphere {h} ======================')
        S = sl.load(os.path.join(analysisDir,'searchlight',f'sl_{h}_s{sn}.h5'))

        # Make the condition_vec with a certain order required for model fitting:
        conditions = []
        # the unimanual conditions:
        lhand_conds = ['lhand:0',  'lhand:60',  'lhand:120',  'lhand:180',  'lhand:240',  'lhand:300']
        rhand_conds = ['rhand:180',  'rhand:120',  'rhand:60',  'rhand:0',  'rhand:300',  'rhand:240']
        conditions.extend(lhand_conds)
        conditions.extend(rhand_conds)

        # the bimanual conditions:
        angles_L = [0, 60, 120, 180, 240, 300]
        angles_R = [180, 120, 60, 0, 300, 240] # intrinsic order for right wrist directions
        if h == 'L':
                # right hand is contralateral:
                for angle_contra in angles_R:
                    for angle_ipsi in angles_L:
                        conditions.append(f'bi:{angle_ipsi}_{angle_contra}')
        if h == 'R':
                # left hand is contralateral:
                for angle_contra in angles_L:
                    for angle_ipsi in angles_R:
                        conditions.append(f'bi:{angle_contra}_{angle_ipsi}')

        # make cond_vec according to the sorting
        cond_vec = np.zeros_like(condition_vec, dtype=int)
        for cond in conditions:
            rows = [j for j, c in enumerate(condition_vec) if c.strip() == cond]
            cond_vec[rows] = conditions.index(cond)

        # run the searchlight:
        results = S.run_parallel(datafiles, model_bimanual_interaction, 
                                {'partition_vec': partition_vec, 'condition_vec': cond_vec, 'hem': h, 
                                'G_contra': G_contra, 'G_incong': G_incong, 'G_bimanual': G_bimanual})
        S.data_to_cifti(results, outfilename=os.path.join(baseDir, 'searchlight', f'glm{glm}_congruency_s{sn}_{h}_cortex.dscalar.nii'))



====================== Processing subject 101, hemisphere L ======================
Loading 481 volumes...
Running searchlight on 32492 centers...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 14 concurrent workers.
[Parallel(n_jobs=-1)]: Done   8 tasks      | elapsed:    1.9s
[Parallel(n_jobs=-1)]: Done  22 tasks      | elapsed:    2.6s
[Parallel(n_jobs=-1)]: Done 428 tasks      | elapsed:    3.9s
[Parallel(n_jobs=-1)]: Done 1328 tasks      | elapsed:    6.1s
[Parallel(n_jobs=-1)]: Done 2228 tasks      | elapsed:    7.4s
[Parallel(n_jobs=-1)]: Done 3328 tasks      | elapsed:    9.1s
[Parallel(n_jobs=-1)]: Done 4428 tasks      | elapsed:   11.1s
[Parallel(n_jobs=-1)]: Done 5728 tasks      | elapsed:   13.5s
[Parallel(n_jobs=-1)]: Done 7028 tasks      | elapsed:   15.8s
[Parallel(n_jobs=-1)]: Done 8528 tasks      | elapsed:   18.3s
[Parallel(n_jobs=-1)]: Done 10028 tasks      | elapsed:   20.8s
[Parallel(n_jobs=-1)]: Done 11728 tasks      | elapsed:   23.1s
[Parallel(n_jobs=-1)]: Done 13428 tasks      | elapsed:   26.3s
[Parallel(n_jobs=-1)]: Done 15328 tasks      | elapsed:   29.4s
[Parallel(n_jobs=-1)]: Done 17228 tasks 

====================== Processing subject 101, hemisphere R ======================
Loading 481 volumes...
Running searchlight on 32492 centers...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 14 concurrent workers.
[Parallel(n_jobs=-1)]: Done   8 tasks      | elapsed:    0.7s
[Parallel(n_jobs=-1)]: Done  22 tasks      | elapsed:    1.7s
[Parallel(n_jobs=-1)]: Done 428 tasks      | elapsed:    3.4s
[Parallel(n_jobs=-1)]: Done 1328 tasks      | elapsed:    4.9s
[Parallel(n_jobs=-1)]: Done 2228 tasks      | elapsed:    6.4s
[Parallel(n_jobs=-1)]: Done 3328 tasks      | elapsed:    8.0s
[Parallel(n_jobs=-1)]: Done 4428 tasks      | elapsed:   10.1s
[Parallel(n_jobs=-1)]: Done 5728 tasks      | elapsed:   12.2s
[Parallel(n_jobs=-1)]: Done 7028 tasks      | elapsed:   14.3s
[Parallel(n_jobs=-1)]: Done 8528 tasks      | elapsed:   16.8s
[Parallel(n_jobs=-1)]: Done 10028 tasks      | elapsed:   19.1s
[Parallel(n_jobs=-1)]: Done 11728 tasks      | elapsed:   21.4s
[Parallel(n_jobs=-1)]: Done 13428 tasks      | elapsed:   24.7s
[Parallel(n_jobs=-1)]: Done 15328 tasks      | elapsed:   27.9s
[Parallel(n_jobs=-1)]: Done 17228 tasks 

====================== Processing subject 104, hemisphere L ======================
Loading 481 volumes...
Running searchlight on 32492 centers...


...

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 14 concurrent workers.
[Parallel(n_jobs=-1)]: Done   8 tasks      | elapsed:    0.8s
[Parallel(n_jobs=-1)]: Done  22 tasks      | elapsed:    1.8s
[Parallel(n_jobs=-1)]: Done 428 tasks      | elapsed:    3.5s
[Parallel(n_jobs=-1)]: Done 1328 tasks      | elapsed:    5.0s
[Parallel(n_jobs=-1)]: Done 2228 tasks      | elapsed:    6.4s
[Parallel(n_jobs=-1)]: Done 3328 tasks      | elapsed:    8.3s
[Parallel(n_jobs=-1)]: Done 4428 tasks      | elapsed:   10.1s
[Parallel(n_jobs=-1)]: Done 5728 tasks      | elapsed:   12.2s
[Parallel(n_jobs=-1)]: Done 7028 tasks      | elapsed:   14.4s
[Parallel(n_jobs=-1)]: Done 8528 tasks      | elapsed:   16.9s
[Parallel(n_jobs=-1)]: Done 10028 tasks      | elapsed:   19.3s
[Parallel(n_jobs=-1)]: Done 11728 tasks      | elapsed:   21.6s
[Parallel(n_jobs=-1)]: Done 13428 tasks      | elapsed:   25.0s
[Parallel(n_jobs=-1)]: Done 15328 tasks      | elapsed:   28.1s
[Parallel(n_jobs=-1)]: Done 17228 tasks 

====================== Processing subject 115, hemisphere L ======================
Loading 481 volumes...
Running searchlight on 32492 centers...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 14 concurrent workers.
[Parallel(n_jobs=-1)]: Done   8 tasks      | elapsed:    0.7s
[Parallel(n_jobs=-1)]: Done  22 tasks      | elapsed:    1.8s
[Parallel(n_jobs=-1)]: Done 428 tasks      | elapsed:    3.4s
[Parallel(n_jobs=-1)]: Done 1328 tasks      | elapsed:    4.9s
[Parallel(n_jobs=-1)]: Done 2228 tasks      | elapsed:    6.4s
[Parallel(n_jobs=-1)]: Done 3328 tasks      | elapsed:    8.3s
[Parallel(n_jobs=-1)]: Done 4428 tasks      | elapsed:   10.2s
[Parallel(n_jobs=-1)]: Done 5728 tasks      | elapsed:   12.3s
[Parallel(n_jobs=-1)]: Done 7028 tasks      | elapsed:   14.4s
[Parallel(n_jobs=-1)]: Done 8528 tasks      | elapsed:   17.0s
[Parallel(n_jobs=-1)]: Done 10028 tasks      | elapsed:   19.5s
[Parallel(n_jobs=-1)]: Done 11728 tasks      | elapsed:   21.7s
[Parallel(n_jobs=-1)]: Done 13428 tasks      | elapsed:   25.1s
[Parallel(n_jobs=-1)]: Done 15328 tasks      | elapsed:   28.3s
[Parallel(n_jobs=-1)]: Done 17228 tasks 

====================== Processing subject 115, hemisphere R ======================
Loading 481 volumes...
Running searchlight on 32492 centers...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 14 concurrent workers.
[Parallel(n_jobs=-1)]: Done   8 tasks      | elapsed:    0.7s
[Parallel(n_jobs=-1)]: Done  22 tasks      | elapsed:    1.8s
[Parallel(n_jobs=-1)]: Done 428 tasks      | elapsed:    3.4s
[Parallel(n_jobs=-1)]: Done 1328 tasks      | elapsed:    4.9s
[Parallel(n_jobs=-1)]: Done 2228 tasks      | elapsed:    6.4s
[Parallel(n_jobs=-1)]: Done 3328 tasks      | elapsed:    8.2s
[Parallel(n_jobs=-1)]: Done 4428 tasks      | elapsed:   10.1s
[Parallel(n_jobs=-1)]: Done 5728 tasks      | elapsed:   12.3s
[Parallel(n_jobs=-1)]: Done 7028 tasks      | elapsed:   14.5s
[Parallel(n_jobs=-1)]: Done 8528 tasks      | elapsed:   17.0s
[Parallel(n_jobs=-1)]: Done 10028 tasks      | elapsed:   19.5s
[Parallel(n_jobs=-1)]: Done 11728 tasks      | elapsed:   21.8s
[Parallel(n_jobs=-1)]: Done 13428 tasks      | elapsed:   25.1s
[Parallel(n_jobs=-1)]: Done 15328 tasks      | elapsed:   28.3s
[Parallel(n_jobs=-1)]: Done 17228 tasks 

### bimanual-specific component

In [ ]:
sn_list = [101, 104, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115]
glm = 1
hem = ['L', 'R']
sl_name = ['left_cortex', 'right_cortex']

def model_bimanual_interaction(data, **kwargs):
    Y_all = data[:-1,:] / np.sqrt(data[-1,:]) # univariate prewhitening
    part_vec = kwargs['partition_vec']
    cond_vec = kwargs['condition_vec']
    G_contra = kwargs['G_contra']
    G_incong = kwargs['G_incong']
    G_bimanual = kwargs['G_bimanual']

    # make PCM dataset object:
    rows_bimanual = [j for j, c in enumerate(cond_vec) if c>11]
    Y_bimanual = Y_all[rows_bimanual,:]
    cond_vec_bimanual = cond_vec[rows_bimanual]
    part_vec_bimanual = part_vec[rows_bimanual]

    G_hat,_ = pcm.est_G_crossval(Y_bimanual,
                                 cond_vec_bimanual,
                                 part_vec_bimanual,
                                 X=pcm.matrix.indicator(part_vec_bimanual))

    # OLS:
    X = np.vstack([
        G_contra.flatten(),
        G_incong.flatten(),
        G_bimanual.flatten(),
    ]).T
    pinvX = np.linalg.pinv(X)
    W = pinvX @ G_hat.flatten()

    # OLS variances:
    ols_contra = W[0] * np.trace(G_contra)/36
    ols_congruency = W[1] * np.trace(G_incong)/36
    ols_bimanual = W[2] * np.trace(G_bimanual)/36

    d_contra = 2/(36-1) * ols_contra*36
    d_congruency = 2/(36-1) * ols_congruency*36
    d_bimanual = 2/(36-1) * ols_bimanual*36

    return d_bimanual   

labels = ['flx_flx',    'flx_flxup',   'flx_extup',   'flx_ext',   'flx_extdn',   'flx_flxdn',
          'flxup_flx',  'flxup_flxup', 'flxup_extup', 'flxup_ext', 'flxup_extdn', 'flxup_flxdn',
          'extup_flx',  'extup_flxup', 'extup_extup', 'extup_ext', 'extup_extdn', 'extup_flxdn',
          'ext_flx',    'ext_flxup',   'ext_extup',   'ext_ext',   'ext_extdn',   'ext_flxdn',
          'extdn_flx',  'extdn_flxup', 'extdn_extup', 'extdn_ext', 'extdn_extdn', 'extdn_flxdn',
          'flxdn_flx',  'flxdn_flxup', 'flxdn_extup', 'flxdn_ext', 'flxdn_extdn', 'flxdn_flxdn']   
F_contra = np.zeros((36,6))
cnd2idx = {'flx':0, 'flxup':1, 'extup':2, 'ext':3, 'extdn':4, 'flxdn':5}
for i in range(36):
    cond_pair = labels[i].split('_')
    cnd_contra = cond_pair[0]
    cnd_ipsi = cond_pair[1]
    idx_contra = cnd2idx[cnd_contra]
    idx_ipsi = cnd2idx[cnd_ipsi]
    F_contra[i, idx_contra] = 1
# center the features:
F_contra -= np.mean(F_contra, axis=0)

# load unimanual G matrix of all regions:
G_uni_df = pd.read_pickle(os.path.join(analysisDir, 'G_uni.pkl'))
# average across all subjects and regions:
G_all = np.concatenate(G_uni_df['G_hat'].tolist(), axis=0)  # (n_regions * N, 6, 6)
Guni = G_all.mean(axis=0)

# build fixed 36x36 model components from group-averaged unimanual G:
G_contra = F_contra @ Guni @ F_contra.T
G_contra = G_contra / np.trace(np.abs(G_contra))

# incongruency model:
ncond = 36
cov_incong = np.zeros((ncond,ncond)) #np.eye(ncond)
idx_incong = [1,2,4,5, 6,9,10,11, 12,15,16,17, 19,20,22,23, 24,25,26,27, 30,31,32,33]
for i in range(len(idx_incong)):
    for j in range(len(idx_incong)):
        if i != j:
            cov_incong[idx_incong[i], idx_incong[j]] = 1
F_incong = np.zeros((ncond,1))
F_incong[idx_incong,0] = 1
cov_incong = F_incong @ F_incong.T
G_incong = pcm.centering(ncond) @ cov_incong @ pcm.centering(ncond) # incongruency model
G_incong = G_incong / np.trace(G_incong) # normalize the overall variance

# bimanual-specific model:
I = np.eye(ncond)
G_bimanual = pcm.centering(ncond) @ I @ I.T @ pcm.centering(ncond) # non-orthogonalized interaction
G_bimanual = G_bimanual / np.trace(G_bimanual) # normalize the overall variance

# orthogonalize the bimanual-specific model:
X = np.c_[G_contra, G_incong]
Io = I - X @ np.linalg.pinv(X) @ I
G_bimanual = pcm.centering(ncond) @ Io @ Io.T @ pcm.centering(ncond) 
G_bimanual = G_bimanual / np.trace(G_bimanual)

for sn in sn_list:
    # try:
    # run marginal MVPA:
    # regressor info:
    reginfo = pd.read_table(os.path.join(baseDir, f'glm{glm}', f's{sn}', 'reginfo.tsv'))
    # get indices of the regressors of interest:
    # regressors_of_interest = reginfo[reginfo.name.str.contains('bi')].index.tolist()
    regressors_of_interest = reginfo.index.tolist()
    
    datafiles = [os.path.join(baseDir, f'glm{glm}', f's{sn}', f'beta_{i+1:04d}.nii') for i in regressors_of_interest]
    datafiles.append(os.path.join(baseDir, f'glm{glm}', f's{sn}', 'ResMS.nii'))
    partition_vec = reginfo.loc[regressors_of_interest, 'run'].values
    condition_vec = reginfo.loc[regressors_of_interest, 'name'].values

    for i, h in enumerate(hem):
        print(f'====================== Processing subject {sn}, hemisphere {h} ======================')
        S = sl.load(os.path.join(analysisDir,'searchlight',f'sl_{h}_s{sn}.h5'))
        
        # Make the condition_vec with a certain order required for model fitting:
        conditions = []
        # the unimanual conditions:
        lhand_conds = ['lhand:0',  'lhand:60',  'lhand:120',  'lhand:180',  'lhand:240',  'lhand:300']
        rhand_conds = ['rhand:180',  'rhand:120',  'rhand:60',  'rhand:0',  'rhand:300',  'rhand:240']
        conditions.extend(lhand_conds)
        conditions.extend(rhand_conds)

        # the bimanual conditions:
        angles_L = [0, 60, 120, 180, 240, 300]
        angles_R = [180, 120, 60, 0, 300, 240] # intrinsic order for right wrist directions
        if h == 'L':
                # right hand is contralateral:
                for angle_contra in angles_R:
                    for angle_ipsi in angles_L:
                        conditions.append(f'bi:{angle_ipsi}_{angle_contra}')
        if h == 'R':
                # left hand is contralateral:
                for angle_contra in angles_L:
                    for angle_ipsi in angles_R:
                        conditions.append(f'bi:{angle_contra}_{angle_ipsi}')

        # make cond_vec according to the sorting
        cond_vec = np.zeros_like(condition_vec, dtype=int)
        for cond in conditions:
            rows = [j for j, c in enumerate(condition_vec) if c.strip() == cond]
            cond_vec[rows] = conditions.index(cond)

        # run the searchlight:
        results = S.run_parallel(datafiles, model_bimanual_interaction, 
                                {'partition_vec': partition_vec, 'condition_vec': cond_vec, 'hem': h, 
                                'G_contra': G_contra, 'G_incong': G_incong, 'G_bimanual': G_bimanual})
        S.data_to_cifti(results, outfilename=os.path.join(baseDir, 'searchlight', f'glm{glm}_bimanual_specific_s{sn}_{h}_cortex.dscalar.nii'))


====================== Processing subject 101, hemisphere L ======================
Loading 481 volumes...
Running searchlight on 32492 centers...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 14 concurrent workers.
[Parallel(n_jobs=-1)]: Done   8 tasks      | elapsed:    0.7s
[Parallel(n_jobs=-1)]: Done  22 tasks      | elapsed:    1.8s
[Parallel(n_jobs=-1)]: Done 428 tasks      | elapsed:    3.4s
[Parallel(n_jobs=-1)]: Done 1328 tasks      | elapsed:    4.9s
[Parallel(n_jobs=-1)]: Done 2228 tasks      | elapsed:    6.4s
[Parallel(n_jobs=-1)]: Done 3328 tasks      | elapsed:    8.1s
[Parallel(n_jobs=-1)]: Done 4428 tasks      | elapsed:   10.0s
[Parallel(n_jobs=-1)]: Done 5728 tasks      | elapsed:   12.1s
[Parallel(n_jobs=-1)]: Done 7028 tasks      | elapsed:   14.3s
[Parallel(n_jobs=-1)]: Done 8528 tasks      | elapsed:   16.7s
[Parallel(n_jobs=-1)]: Done 10028 tasks      | elapsed:   19.2s
[Parallel(n_jobs=-1)]: Done 11728 tasks      | elapsed:   21.5s
[Parallel(n_jobs=-1)]: Done 13428 tasks      | elapsed:   24.7s
[Parallel(n_jobs=-1)]: Done 15328 tasks      | elapsed:   27.9s
[Parallel(n_jobs=-1)]: Done 17228 tasks 

====================== Processing subject 101, hemisphere R ======================
Loading 481 volumes...
Running searchlight on 32492 centers...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 14 concurrent workers.
[Parallel(n_jobs=-1)]: Done   8 tasks      | elapsed:    0.7s
[Parallel(n_jobs=-1)]: Done  22 tasks      | elapsed:    1.8s
[Parallel(n_jobs=-1)]: Done 428 tasks      | elapsed:    3.3s
[Parallel(n_jobs=-1)]: Done 1328 tasks      | elapsed:    4.8s
[Parallel(n_jobs=-1)]: Done 2228 tasks      | elapsed:    6.3s
[Parallel(n_jobs=-1)]: Done 3328 tasks      | elapsed:    7.9s
[Parallel(n_jobs=-1)]: Done 4428 tasks      | elapsed:    9.9s
[Parallel(n_jobs=-1)]: Done 5728 tasks      | elapsed:   12.0s
[Parallel(n_jobs=-1)]: Done 7028 tasks      | elapsed:   14.1s
[Parallel(n_jobs=-1)]: Done 8528 tasks      | elapsed:   16.5s
[Parallel(n_jobs=-1)]: Done 10028 tasks      | elapsed:   18.8s
[Parallel(n_jobs=-1)]: Done 11728 tasks      | elapsed:   21.1s
[Parallel(n_jobs=-1)]: Done 13428 tasks      | elapsed:   24.4s
[Parallel(n_jobs=-1)]: Done 15328 tasks      | elapsed:   27.4s
[Parallel(n_jobs=-1)]: Done 17228 tasks 

====================== Processing subject 104, hemisphere L ======================
Loading 481 volumes...
Running searchlight on 32492 centers...


...

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 14 concurrent workers.
[Parallel(n_jobs=-1)]: Done   8 tasks      | elapsed:    0.7s
[Parallel(n_jobs=-1)]: Done  22 tasks      | elapsed:    1.8s
[Parallel(n_jobs=-1)]: Done 428 tasks      | elapsed:    3.5s
[Parallel(n_jobs=-1)]: Done 1328 tasks      | elapsed:    5.0s
[Parallel(n_jobs=-1)]: Done 2228 tasks      | elapsed:    6.4s
[Parallel(n_jobs=-1)]: Done 3328 tasks      | elapsed:    8.2s
[Parallel(n_jobs=-1)]: Done 4428 tasks      | elapsed:   10.1s
[Parallel(n_jobs=-1)]: Done 5728 tasks      | elapsed:   12.2s
[Parallel(n_jobs=-1)]: Done 7028 tasks      | elapsed:   14.4s
[Parallel(n_jobs=-1)]: Done 8528 tasks      | elapsed:   16.8s
[Parallel(n_jobs=-1)]: Done 10028 tasks      | elapsed:   19.3s
[Parallel(n_jobs=-1)]: Done 11728 tasks      | elapsed:   21.5s
[Parallel(n_jobs=-1)]: Done 13428 tasks      | elapsed:   24.8s
[Parallel(n_jobs=-1)]: Done 15328 tasks      | elapsed:   28.0s
[Parallel(n_jobs=-1)]: Done 17228 tasks 

====================== Processing subject 115, hemisphere L ======================
Loading 481 volumes...
Running searchlight on 32492 centers...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 14 concurrent workers.
[Parallel(n_jobs=-1)]: Done   8 tasks      | elapsed:    0.7s
[Parallel(n_jobs=-1)]: Done  22 tasks      | elapsed:    1.8s
[Parallel(n_jobs=-1)]: Done 428 tasks      | elapsed:    3.4s
[Parallel(n_jobs=-1)]: Done 1328 tasks      | elapsed:    4.9s
[Parallel(n_jobs=-1)]: Done 2228 tasks      | elapsed:    6.4s
[Parallel(n_jobs=-1)]: Done 3328 tasks      | elapsed:    8.2s
[Parallel(n_jobs=-1)]: Done 4428 tasks      | elapsed:   10.1s
[Parallel(n_jobs=-1)]: Done 5728 tasks      | elapsed:   12.3s
[Parallel(n_jobs=-1)]: Done 7028 tasks      | elapsed:   14.5s
[Parallel(n_jobs=-1)]: Done 8528 tasks      | elapsed:   17.0s
[Parallel(n_jobs=-1)]: Done 10028 tasks      | elapsed:   19.5s
[Parallel(n_jobs=-1)]: Done 11728 tasks      | elapsed:   21.7s
[Parallel(n_jobs=-1)]: Done 13428 tasks      | elapsed:   25.1s
[Parallel(n_jobs=-1)]: Done 15328 tasks      | elapsed:   28.2s
[Parallel(n_jobs=-1)]: Done 17228 tasks 

====================== Processing subject 115, hemisphere R ======================
Loading 481 volumes...
Running searchlight on 32492 centers...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 14 concurrent workers.
[Parallel(n_jobs=-1)]: Done   8 tasks      | elapsed:    0.7s
[Parallel(n_jobs=-1)]: Done  22 tasks      | elapsed:    1.7s
[Parallel(n_jobs=-1)]: Done 428 tasks      | elapsed:    3.4s
[Parallel(n_jobs=-1)]: Done 1328 tasks      | elapsed:    4.9s
[Parallel(n_jobs=-1)]: Done 2228 tasks      | elapsed:    6.4s
[Parallel(n_jobs=-1)]: Done 3328 tasks      | elapsed:    8.2s
[Parallel(n_jobs=-1)]: Done 4428 tasks      | elapsed:   10.0s
[Parallel(n_jobs=-1)]: Done 5728 tasks      | elapsed:   12.2s
[Parallel(n_jobs=-1)]: Done 7028 tasks      | elapsed:   14.3s
[Parallel(n_jobs=-1)]: Done 8528 tasks      | elapsed:   16.8s
[Parallel(n_jobs=-1)]: Done 10028 tasks      | elapsed:   19.4s
[Parallel(n_jobs=-1)]: Done 11728 tasks      | elapsed:   21.6s
[Parallel(n_jobs=-1)]: Done 13428 tasks      | elapsed:   25.0s
[Parallel(n_jobs=-1)]: Done 15328 tasks      | elapsed:   28.1s
[Parallel(n_jobs=-1)]: Done 17228 tasks 

### encoding profile

In [ ]:
import matplotlib.path as mpath

# DEFINE PATCH:
num_subj = 12
    
    # (119.53, 104),
    # (124.41, 71.85)
# Example: [(x1, y1), (x2, y2), (x3, y3), ...]
polygon_coords = [
    (-25, 41.4), 
    (33.8, 37.2),
    (76.9, 64.2),
    (122.3,73.3),
    (106,110),
    (-44.2, 89.2)
]
patch_path = mpath.Path(polygon_coords)

# Define the axis along which you will create bins
x1, x2 = -25, 122.3
dx = 3
ROIs = ['PMd', 'M1', 'S1', 'SPLa', 'SPLp']
ROI_Xcoords = [-25, 0.8, 16, 45.6, 74, 122.3]

surfPy_dir = '/Users/aghavamp/Desktop/Projects/surfAnalysisPy/standard_mesh'
surf_path = os.path.abspath(os.path.join(surfPy_dir, f'fs_L', f'fs_LR.32k.Lm.flat.surf.gii'))
# Load GIFTI surface
gii = nb.load(surf_path)

# load data:
data = nb.load(os.path.join(analysisDir, 'psc.dscalar.nii'))
beta = data.get_fdata()  # shape: (N, 1) for flat surfaces
scalar_axis = data.header.get_axis(0)  # scalar axis for flat surfaces
row_names = [ax[0] for ax in scalar_axis]  # get row names from scalar axis
idx_lhand = [i for i in range(len(row_names)) if 'lhand' in row_names[i]][0:num_subj]
idx_rhand = [i for i in range(len(row_names)) if 'rhand' in row_names[i]][0:num_subj]
idx_bi = [i for i in range(len(row_names)) if 'bi' in row_names[i]][0:num_subj]

# Get vertices (XYZ)
verts = gii.agg_data('pointset')   # shape: (N, 3); for flat, Z should be 0
x = verts[:, 0]
y = verts[:, 1]
print(len(verts))

# Check which vertices are inside the polygon
points_to_check = np.vstack((x, y)).T
in_strip = patch_path.contains_points(points_to_check)

# You can visualize the selected patch if you want
# fig, ax = plt.subplots(figsize=(3, 2.5))
# plt.scatter(x[in_strip], y[in_strip], s=1)
# plt.show()

bins = np.arange(x1, x2, dx)
centers = 0.5 * (bins[:-1] + bins[1:])

psc_profiles = {'lhand': np.full((num_subj, len(bins) - 1), np.nan),
                'rhand': np.full((num_subj, len(bins) - 1), np.nan),
                'bi': np.full((num_subj, len(bins) - 1), np.nan)}

# profile = np.full((num_subj, len(bins) - 1), np.nan)
# loop on subjects:
for sn in range(num_subj):
    beta_sn = beta[idx_bi[sn], 0:len(verts)] # left hem data

    # Calculate mean PSC in bins for Bimanual:
    for i in range(len(bins) - 1):
        b0, b1 = bins[i], bins[i+1]
        mbin = in_strip & (x >= b0) & (x < b1)
        psc_profiles['bi'][sn, i] = np.nanmean(beta_sn[mbin])  

    # Calculate mean PSC in bins for lhand:
    beta_sn = beta[idx_lhand[sn], 0:len(verts)] # left hem data
    for i in range(len(bins) - 1):
        b0, b1 = bins[i], bins[i+1]
        mbin = in_strip & (x >= b0) & (x < b1)
        psc_profiles['lhand'][sn, i] = np.nanmean(beta_sn[mbin])
        
    # Calculate mean PSC in bins for rhand:
    beta_sn = beta[idx_rhand[sn], 0:len(verts)] # right hem data
    for i in range(len(bins) - 1):
        b0, b1 = bins[i], bins[i+1]
        mbin = in_strip & (x >= b0) & (x < b1)
        psc_profiles['rhand'][sn, i] = np.nanmean(beta_sn[mbin])
        
avg_psc_bi = np.nanmean(psc_profiles['bi'], axis=0)
sem_psc_bi = np.nanstd(psc_profiles['bi'], axis=0) / np.sqrt(num_subj)
avg_psc_lhand = np.nanmean(psc_profiles['lhand'], axis=0)
sem_psc_lhand = np.nanstd(psc_profiles['lhand'], axis=0) / np.sqrt(num_subj)
avg_psc_rhand = np.nanmean(psc_profiles['rhand'], axis=0)
sem_psc_rhand = np.nanstd(psc_profiles['rhand'], axis=0) / np.sqrt(num_subj)

# Plotting the results
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 7
fig, ax = plt.subplots(figsize=(2, 1.4))
# ax.plot(centers, avg_psc_bi, label='Bimanual', color='k', linewidth=1)
ax.plot(centers, avg_psc_lhand, label='ipsi', color='#0072B2', linewidth=1)
ax.plot(centers, avg_psc_rhand, label='contra', color='#F8766D', linewidth=1)

# ax.fill_between(centers, avg_psc_bi - sem_psc_bi, avg_psc_bi + sem_psc_bi, color='k', alpha=0.2)
ax.fill_between(centers, avg_psc_lhand - sem_psc_lhand, avg_psc_lhand + sem_psc_lhand, color='#0072B2', alpha=0.2)
ax.fill_between(centers, avg_psc_rhand - sem_psc_rhand, avg_psc_rhand + sem_psc_rhand, color='#F8766D', alpha=0.2)
ax.set_title('Left Hem')
for roi_x in ROI_Xcoords:
    ax.axvline(x=roi_x, color='k', linestyle=':', label='_ignore_', alpha=0.5)
ax.axhline(y=0, color='k', linestyle='-', linewidth=1, alpha=0.5)
# ax.set_xlabel('X Coordinate (mm)')
ax.set_xticklabels([''])
# ax.set_ylabel('Mean PSC')
ax.set_ylim((0, 1.2))
ax.set_yticks(np.arange(0, 1.5, 0.5))
# ax.legend()
plt.tight_layout()
plt.savefig(f'../figures/PSC/PSC_profile_leftHemisphere.pdf', bbox_inches="tight")
plt.show()

